In [13]:
logs = pd.read_csv("server_logs.csv")

In [14]:
print(retail.head())
print(employee.head())
print(logs.head())

   OrderID CustomerID ProductCategory  SalesAmount Region   OrderDate  \
0        1       C101     Electronics       5000.0  South  2023-01-01   
1        2       C102     Electronics       5000.0  South  2023-01-02   
2        3       C103       Furniture       3000.0  North  2023-01-03   
3        4       C104     Electronics       5000.0  South  2023-01-04   
4        5       C105       Furniture       2000.0  South  2023-01-05   

  Imputation_Method  
0   Regional_Median  
1          Original  
2          Original  
3   Regional_Median  
4          Original  
            Employee_Name
0  Dr. Rahul Kumar Sharma
1                Priya M.
2        Mr. S. Venkatesh
3        Ms. Anjali Singh
4            Rahul Sharma
          Timestamp UserID   Action  ResponseTime
0  2023-01-01 10:00     U1    ERROR           120
1  2023-01-01 10:01     U2  SUCCESS            50
2  2023-01-01 10:02     U1    ERROR           200
3  2023-01-01 10:03     U3    ERROR           300
4  2023-01-01 10:04    

In [15]:
# =====================
# QUESTION 1
# =====================

import pandas as pd

# Load dataset (make sure file name is correct)
retail = pd.read_csv("retail_sales.csv")

# Function to fill missing values
def fill_values(row):
    if pd.isna(row['SalesAmount']):

        # Median of same ProductCategory + Region
        median_region = retail[
            (retail['ProductCategory'] == row['ProductCategory']) &
            (retail['Region'] == row['Region'])
        ]['SalesAmount'].median()

        if not pd.isna(median_region):
            return pd.Series([median_region, "Regional_Median"])

        else:
            # Median of ProductCategory
            median_cat = retail[
                retail['ProductCategory'] == row['ProductCategory']
            ]['SalesAmount'].median()

            return pd.Series([median_cat, "Category_Median"])

    # If value already exists
    return pd.Series([row['SalesAmount'], "Original"])

# Apply function
retail[['SalesAmount', 'Imputation_Method']] = retail.apply(fill_values, axis=1)

# Show result
print(retail)

# Save cleaned file
retail.to_csv("cleaned_retail_sales.csv", index=False)

# Count methods used
print("\nImputation Count:")
print(retail['Imputation_Method'].value_counts())

   OrderID CustomerID ProductCategory  SalesAmount Region   OrderDate  \
0        1       C101     Electronics       5000.0  South  2023-01-01   
1        2       C102     Electronics       5000.0  South  2023-01-02   
2        3       C103       Furniture       3000.0  North  2023-01-03   
3        4       C104     Electronics       5000.0  South  2023-01-04   
4        5       C105       Furniture       2000.0  South  2023-01-05   
5        6       C106     Electronics       7000.0  North  2023-01-06   

  Imputation_Method  
0   Regional_Median  
1          Original  
2          Original  
3   Regional_Median  
4          Original  
5          Original  

Imputation Count:
Imputation_Method
Original           4
Regional_Median    2
Name: count, dtype: int64


In [17]:
from google.colab import files
files.download("cleaned_retail_sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# =====================
# QUESTION 2
# =====================

import pandas as pd
import re

# Function to clean and split names
def clean_name(name):
    # Remove titles
    name = re.sub(r"(Dr\.|Mr\.|Ms\.)", "", name).strip()

    parts = name.split()

    first = parts[0]
    last = parts[-1]

    # Middle initial (if exists)
    middle = parts[1][0] if len(parts) == 3 else ""

    return pd.Series([first, middle, last])

# Apply function
employee[['First_Name', 'Middle_Initial', 'Last_Name']] = employee['Employee_Name'].apply(clean_name)

# Create username: firstinitial.lastname
employee['Username'] = (employee['First_Name'].str[0] + "." + employee['Last_Name']).str.lower()

# Handle duplicate usernames
counts = {}
duplicates = 0
new_usernames = []

for u in employee['Username']:
    if u in counts:
        counts[u] += 1
        new_usernames.append(f"{u}{counts[u]}")
        duplicates += 1
    else:
        counts[u] = 1
        new_usernames.append(u)

employee['Username'] = new_usernames

# Output
print(employee.head(8))
print("\nTotal duplicate usernames handled:", duplicates)

            Employee_Name First_Name Middle_Initial  Last_Name     Username
0  Dr. Rahul Kumar Sharma      Rahul              K     Sharma     r.sharma
1                Priya M.      Priya                        M.         p.m.
2        Mr. S. Venkatesh         S.                 Venkatesh  s.venkatesh
3        Ms. Anjali Singh     Anjali                     Singh      a.singh
4            Rahul Sharma      Rahul                    Sharma    r.sharma2

Total duplicate usernames handled: 1


In [19]:
# =====================
# QUESTION 3
# =====================

import pandas as pd

# Dictionaries to store total time and count
total_time = {}
count = {}

# Read file in chunks (small size for demo)
for chunk in pd.read_csv("server_logs.csv", chunksize=2):

    # Select required columns
    chunk = chunk[['UserID', 'Action', 'ResponseTime']]

    # Filter only ERROR records
    error_data = chunk[chunk['Action'] == 'ERROR']

    # Loop through rows
    for index, row in error_data.iterrows():
        user = row['UserID']
        time = row['ResponseTime']

        if user in total_time:
            total_time[user] += time
            count[user] += 1
        else:
            total_time[user] = time
            count[user] = 1

# Calculate average response time
avg_time = {}
for user in total_time:
    avg_time[user] = total_time[user] / count[user]

# Sort and get top users
top_users = sorted(avg_time.items(), key=lambda x: x[1], reverse=True)

# Print result
print("Top Users with Highest Average Response Time:\n")

for user, avg in top_users:
    print(user, ":", avg)

Top Users with Highest Average Response Time:

U3 : 300.0
U1 : 160.0
U2 : 150.0
